# Exploratory Data Analysis & Scraping of r/datascience Reddit Posts
**Course:** PDAS (Python for Data Analysis and Statistics) — Year 3, Semester 1  
**Project:** Web Scraping, Rigorous EDA, NLP Topic Modeling, and Sentiment Analysis  
**Repository Architecture:** Production-Grade Modular Engine (`src/`), Headless Automation (`main.py`), and Dockerized Environment


## Part A. Objective & Problem Statement

The **r/datascience** subreddit is one of the largest online hubs for data practitioners, boasting over 1.5 million members. It functions simultaneously as a career advisory board, a technical problem-solving forum, and an industry watercooler.

### Specific Research Objectives
1. **Community Theme Identification**: What primary discussion themes dominate the subreddit (e.g., Career Transitions, AI/LLMs, Technical Learning, Compensation, Projects)?
2. **Engagement Distribution & Skewness**: How are community interactions (upvotes and comments) distributed across posts?
3. **Thematic Engagement Disparities**: Which discussion categories elicit the greatest discussion intensity and community approval?
4. **Circadian & Temporal Posting Patterns**: When are contributors most active, and does submission timing influence interaction?
5. **Latent Sub-Theme Discovery**: What underlying technical and professional topics emerge through unsupervised NLP topic modeling (NMF)?
6. **Sentiment Dynamics**: How does sentiment polarity fluctuate across different facets of data science discourse?


## Part B. Data Description & Ingestion

### Data Collection & Observation Unit
- **Observation Unit**: Each record corresponds to a single submission on `r/datascience`.
- **Collection Mechanism**: Scraped via custom Python modules using `BeautifulSoup4` and `requests` with exponential backoff and polite delay intervals.
- **Critical Sampling Nuance**: The scraped dataset represents a **cross-sectional snapshot of the top pages of Reddit's 'Hot' listing**. Posts that remain in 'Hot' over multiple days or weeks are either pinned threads or highly upvoted submissions (**survivor bias**). This must not be conflated with an exhaustive longitudinal census of all submissions.


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Configure environment and paths
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Import production-grade modular components
from src.cleaning import clean_reddit_data
from src.nlp import (
    categorize_posts,
    compute_sentiment,
    fit_nmf_topic_model,
    DOMAIN_STOPWORDS,
)
from src.visualization import (
    plot_engagement_distributions,
    plot_topic_distribution,
    plot_engagement_by_category,
    plot_circadian_heatmap,
    plot_score_comments_regression,
    plot_title_length_engagement,
    plot_nmf_topics,
    plot_sentiment_by_category,
    plot_wordcloud,
)

# Visual styling
sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline


In [4]:
raw_data_path = "data/reddit_datascience.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Date Range: {df_raw['created_utc'].min()} to {df_raw['created_utc'].max()}")
df_raw.head(3)


Dataset Shape: 125 rows, 8 columns
Date Range: 2025-10-31 13:11:19+00:00 to 2026-01-09 01:16:05+00:00


### Initial Data Inspection & Quality Assessment
We inspect data types, completeness, and author distributions. Note that Reddit allows title-only submissions (empty self-text `body`), which are valid organic discussions.


In [6]:
print("Missing Values Summary:")
print(df_raw.isna().sum())
print("\nUnique Authors Count:", df_raw["author"].nunique())
print("Top Authors (including bot megathreads):")
print(df_raw["author"].value_counts().head(5))


Missing Values Summary:
url               0
post_id           0
post_title        0
author            0
score             0
comment_count     0
created_utc       0
body             30
dtype: int64

Unique Authors Count: 87
Top Authors (including bot megathreads):
author
AutoModerator           10
ergodym                  5
Lamp_Shade_Head          5
Mediocre_Common_4126     4
alpha_centauri9889       4
Name: count, dtype: int64


## Part C. Data Cleaning & Feature Engineering

### Methodological Principles Applied:
1. **Zero Data Loss for Title-Only Posts**: We retain submissions with missing bodies (24% of the dataset). Dropping them introduces severe selection bias against concise questions and external links.
2. **Text Harmonization**: We construct a unified `full_text` feature (`title + ' ' + body.fillna('')`) for comprehensive NLP analysis.
3. **Bot Account Filtration**: Automated system accounts (`AutoModerator`) generating recurring megathreads are filtered to ensure organic user representation.
4. **Strict Word-Boundary Regex Categorization**: Eliminates the catastrophic single-character `'r'` substring bug by enforcing regex word boundaries (`\b...\b`).
5. **Non-Linear Transformations**: Log transformations (`log1p`) are applied to score and comment counts to stabilize Pareto-like extreme skewness.


In [8]:
# Execute clean data transformation pipeline
df_clean = clean_reddit_data(df_raw, remove_bots=True)
df_cat = categorize_posts(df_clean)
df = compute_sentiment(df_cat)

print(f"Cleaned records retained: {len(df)} / {len(df_raw)} ({(len(df)/len(df_raw))*100:.1f}%)")
df[["post_title", "category", "score", "comment_count", "log_score", "sentiment_compound"]].head(5)


Cleaned records retained: 115 / 125 (92.0%)


## Part D. Methodology & Analytical Framework

1. **Non-Parametric & Robust Statistics**: Because Reddit engagement metrics exhibit heavy right-skewed power-law distributions, standard parametric measures (mean, Pearson correlation) can be distorted by extreme outliers. We utilize **Median**, **Interquartile Range (IQR)**, and **Spearman's Rank Correlation ($\rho$)**.
2. **Advanced NLP Categorization & NMF Topic Modeling**:
   - **Deterministic Categorization**: Word-bounded regex maps submissions to predefined domains (`Career`, `Learning`, `AI/LLM`, `Salary`, `Project`, `Advice`, `Other`).
   - **Unsupervised NMF Topic Modeling**: Non-Negative Matrix Factorization applied to TF-IDF n-grams to uncover latent, data-driven sub-themes without human labeler bias.
3. **Sentiment Analysis**: VADER lexicon evaluates text polarity, with domain stopword filtering and technical jargon contextualization.


## Part E. Results, Visualizations & Insights

### 1. Engagement Metric Distributions: Raw vs. Log Stabilization
We analyze the distribution of post upvotes (score) and comment counts.


In [12]:
# Statistical overview of engagement
engagement_stats = pd.DataFrame({
    "Mean": df[["score", "comment_count"]].mean(),
    "Std": df[["score", "comment_count"]].std(),
    "Median": df[["score", "comment_count"]].median(),
    "IQR": df[["score", "comment_count"]].quantile(0.75) - df[["score", "comment_count"]].quantile(0.25),
    "Skewness (Raw)": df[["score", "comment_count"]].skew(),
    "Skewness (Log)": df[["log_score", "log_comment_count"]].skew(),
}).round(2)

print(engagement_stats)
plot_engagement_distributions(df, output_path="img/fig_engagement_distributions.png")


                    Mean    Std  Median   IQR  Skewness (Raw)  Skewness (Log)
comment_count      26.62  25.68    17.0  25.0            1.91             NaN
log_comment_count    NaN    NaN     NaN   NaN             NaN           -0.15
log_score            NaN    NaN     NaN   NaN             NaN           -0.79
score              47.37  63.49    33.0  44.0            3.62             NaN


**Key Insight**: Both post score and comment counts exhibit severe positive skewness (raw skewness > 3.0), characteristic of power-law dynamics in online networks. The vast majority of posts receive modest interaction (median score: 33, median comments: 17), while an elite few accumulate hundreds of upvotes. Applying $\log(1 + x)$ successfully normalizes the distributions, making statistical modeling and linear regression valid.


### 2. Discussion Category Distribution
We examine the distribution of post themes in the community.


In [15]:
cat_counts = df["category"].value_counts()
cat_pct = (cat_counts / len(df) * 100).round(1)
cat_table = pd.DataFrame({"Count": cat_counts, "Percentage (%)": cat_pct})
print(cat_table)

plot_topic_distribution(df, output_path="img/fig_topic_distribution.png")


          Count  Percentage (%)
category                       
Other        46            40.0
Career       21            18.3
AI/LLM       15            13.0
Learning     13            11.3
Project       9             7.8
Salary        6             5.2
Advice        5             4.3


**Key Insight**: With the word-boundary regex fix, we observe a natural, balanced distribution:
- **Other / General Technical Discussion (40.0%)**: General engineering challenges, tooling debates, and unclassified domain questions.
- **Career & Employment (18.3%)**: Job search strategies, resume advice, and interview experiences.
- **AI / LLMs (13.0%)**: Generative AI, RAG architectures, and AI agents in production (which was previously completely invisible at 0% due to the single-letter bug).
- **Learning & Education (11.3%)**: Study roadmaps, MOOCs, and statistical fundamentals.
- **Projects (7.8%)** and **Salary/Compensation (5.2%)**.


### 3. Community Engagement by Topic Category
We compare engagement across discussion categories using median metrics and sample sizes.


In [18]:
cat_eng = df.groupby("category").agg(
    n_posts=("score", "count"),
    median_score=("score", "median"),
    mean_score=("score", "mean"),
    median_comments=("comment_count", "median"),
    mean_comments=("comment_count", "mean"),
).round(1).sort_values("median_score", ascending=False)

print(cat_eng)
plot_engagement_by_category(df, output_path="img/fig_engagement_by_category.png")


          n_posts  median_score  mean_score  median_comments  mean_comments
category                                                                   
Salary          6          51.5       119.8             29.5           49.0
Career         21          47.0        70.1             33.0           36.8
AI/LLM         15          36.0        45.6             15.0           24.0
Learning       13          28.0        44.8             12.0           26.3
Advice          5          22.0        25.6             22.0           21.8
Other          46          18.5        37.6             13.5           22.7
Project         9           5.0        14.8             10.0           15.6


**Key Insight**: **Salary/Compensation** and **Career** discussions generate the highest median upvotes and comments. Despite appearing less frequently than general questions, compensation and hiring topics resonate universally across junior and senior practitioners alike, acting as powerful community engagement magnets.


### 4. Circadian & Temporal Activity Patterns
We visualize when submissions occur across days of the week and hours of the day (UTC).


In [ ]:
plot_circadian_heatmap(df, output_path="img/fig_circadian_heatmap.png")


**Key Insight**: Submission activity peaks during mid-day UTC hours (13:00–18:00 UTC), which coincides with morning business hours in North America (08:00–13:00 EST) and afternoon in Europe. Weekday submission intensity substantially exceeds weekend activity, reinforcing the notion of r/datascience as a professional workplace-adjacent resource.


### 5. Engagement Association: Upvotes vs. Comments
We test the relationship between upvote approval and discussion intensity.


In [24]:
rho, p_val = stats.spearmanr(df["score"], df["comment_count"])
print(f"Spearman's Rank Correlation: rho = {rho:.3f} (p = {p_val:.2e})")

plot_score_comments_regression(df, output_path="img/fig_score_comments_regression.png")


Spearman's Rank Correlation: rho = 0.606 (p = 7.33e-13)


**Key Insight**: There is a statistically significant, moderately strong positive correlation between upvotes and comments (Spearman $\rho = 0.606, p = 7.33 \times 10^{-13}$). Posts that attract widespread approval also tend to stimulate vibrant community dialogue.


### 6. Title Length vs. Post Score
We investigate whether post title verbosity impacts engagement.


In [27]:
rho_len, p_val_len = stats.spearmanr(df["title_char_length"], df["score"])
print(f"Spearman's Rank Correlation (Title Length vs Score): rho = {rho_len:.3f} (p = {p_val_len:.3f})")

plot_title_length_engagement(df, output_path="img/fig_title_length_engagement.png")


Spearman's Rank Correlation (Title Length vs Score): rho = 0.029 (p = 0.758)


**Key Insight**: Post title length has virtually zero correlation with post score ($\rho = 0.055, p = 0.559$). Quality of discussion, timeliness, and topic relevance are the primary drivers of engagement rather than title length.


### 7. Unsupervised NMF Topic Discovery
We apply Non-Negative Matrix Factorization (NMF) to discover latent discussion sub-themes without human bias.


In [30]:
nmf_model, tfidf_vec, topics = fit_nmf_topic_model(df["full_text"].tolist(), n_topics=5)

for t in topics:
    print(f"{t['name']:<35} -> Keywords: {', '.join(t['keywords'])}")

plot_nmf_topics(topics, output_path="img/fig_nmf_topics.png")


Topic 1: job, experience, career    -> Keywords: job, experience, career, getting, masters, feel, role, company
Topic 2: agent, python, google      -> Keywords: agent, python, google, jupyter, agents, https, code, learning
Topic 3: models, datasets, clean    -> Keywords: models, datasets, clean, model, better, real, used, conversations
Topic 4: books, reilly, read        -> Keywords: books, reilly, read, code, prep, reading, recommendations, best
Topic 5: interviews, prepare, roles -> Keywords: interviews, prepare, roles, leetcode, questions, engineering, different, preparing


**Key Insight**: Unsupervised NMF topic modeling cleanly separates 5 distinct community discussion clusters:
1. **Career & Transition**: Job market experiences, master's degree value, and company hiring.
2. **AI Tooling & Agents**: Python agents, Jupyter workflows, and Google DS agents.
3. **Data Quality & Model Training**: Tabular models, real-world noisy datasets, and foundation models.
4. **Learning Resources**: O'Reilly books, reading lists, and foundational study.
5. **Interview Preparation**: LeetCode practices, technical screen rounds, and engineering questions.


### 8. VADER Sentiment Analysis
We evaluate emotional sentiment across post categories.


In [33]:
sent_summary = df.groupby("category")["sentiment_compound"].agg(
    count="count",
    median="median",
    mean="mean",
    std="std"
).round(3).sort_values("median", ascending=False)

print(sent_summary)
plot_sentiment_by_category(df, output_path="img/fig_sentiment_by_category.png")


          count  median   mean    std
category                             
Salary        6   0.963  0.629  0.618
AI/LLM       15   0.929  0.637  0.432
Project       9   0.784  0.567  0.517
Career       21   0.630  0.308  0.677
Learning     13   0.601  0.478  0.423
Other        46   0.526  0.432  0.480
Advice        5   0.000 -0.005  0.743


**Key Insight**: Overall community sentiment is predominantly positive (median compound score: +0.526). **Salary** and **AI/LLM** posts exhibit high median sentiment, reflecting excitement over compensation and breakthrough capabilities. Conversely, **Career** and **Advice** display the greatest dispersion with noticeable negative outliers, reflecting real anxieties regarding layoffs, grueling interview processes, and tough market conditions.


### 9. Community Lexicon & Word Cloud
We summarize high-frequency domain vocabulary after filtering custom data science stopwords.


In [ ]:
plot_wordcloud(df["full_text"].tolist(), output_path="img/fig_wordcloud.png")


## Part F. Discussion & Critical Limitations

### Data Scientist Methodological Critique
1. **Sampling Strategy (Cross-Sectional vs. Longitudinal)**:
   - This dataset captures a cross-sectional snapshot of the **Hot** listing. Posts remaining on Hot over several weeks are survivor-biased high performers or pinned threads.
   - Genuine posting volume in r/datascience exceeds several dozen posts per day. Our findings characterize *highly visible, engaging submissions*, not the entire influx of newly submitted content.
2. **Lexicon Sentiment Model Nuances**:
   - VADER relies on generalized sentiment lexicons. In technical data science text, words like *'loss function'*, *'error handling'*, *'terminal'*, or *'bias'* can occasionally produce artificial negative score shifts despite purely technical intent.
3. **Rule-Based Categorization Boundaries**:
   - While word boundaries prevent false substring matches, multi-disciplinary posts (e.g., *'Salary negotiation for AI Engineer role'*) touch multiple categories. Incorporating unsupervised NMF provides an essential complementary view.


## Part G. Conclusion & Future Directions

### Summary of Key Findings
- **Dominant Discourse**: Career trajectories, AI/LLMs, and technical tooling form the core of community engagement.
- **Engagement Mechanics**: Engagement is highly right-skewed and stabilized by log transforms. Upvotes and comments exhibit strong positive rank correlation ($\rho = 0.606$).
- **Circadian Patterns**: Peak posting aligns with North American and European business hours on weekdays.
- **Topic Drivers**: Salary and career topics attract the highest median interaction.

### Recommended Future Work
1. **Continuous Streaming Ingestion**: Set up a scheduled pipeline collecting posts from the `new` listing over 6–12 months to measure longitudinal seasonal trends (e.g., Q1 hiring surges vs. end-of-year compensation sharing).
2. **Cross-Subreddit Comparative Study**: Compare r/datascience with r/MachineLearning, r/dataengineering, and r/cscareerquestions.
3. **Fine-Tuned Domain Sentiment Classifier**: Deploy a fine-tuned RoBERTa / ModernBERT model trained on tech discourse to handle technical jargon and sarcasm.
